In [29]:
import pandas as pd
import tensorflow as tf
import matplotlib as pl
import tensorflow_decision_forests as tfdf
import numpy as np
import keras
from keras import layers
from sklearn.model_selection import train_test_split


print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print("Built with CUDA:", tf.test.is_built_with_cuda())

Num GPUs Available:  0
Built with CUDA: True


In [ ]:
df = pd.read_csv('Datos/santiago_historico_limpio.csv')
df = df.dropna(subset=['price'])
df = df.drop(['id'], axis = 1)

train_df = df.sample(frac=0.8, random_state=42)
test_df = df.drop(train_df.index)


train_df = tfdf.keras.pd_dataframe_to_tf_dataset(
    train_df,
    task = tfdf.keras.Task.REGRESSION,
    label = 'price',
)

test_df = tfdf.keras.pd_dataframe_to_tf_dataset(
    test_df,
    task = tfdf.keras.Task.REGRESSION,
    label = 'price',
)

train_df = train_df.prefetch(tf.data.AUTOTUNE)

In [14]:
len(test_df)

36

# Gradient Boost 
https://www.tensorflow.org/decision_forests/api_docs/python/tfdf/keras/GradientBoostedTreesModel

In [21]:
model = tfdf.keras.GradientBoostedTreesModel(
    task = tfdf.keras.Task.REGRESSION,
    num_trees = 500,
    max_depth = 6,
    growing_strategy = "BEST_FIRST_GLOBAL",
    shrinkage = 0.1,
    l2_regularization = 0.1
)
model.fit(train_df)

Use /tmp/tmp5_gmw3vn as temporary training directory
Reading training dataset...
Training dataset read in 0:00:00.857102. Found 143074 examples.
Training model...


2026-05-09 19:08:41.093547: W external/ydf/yggdrasil_decision_forests/learner/gradient_boosted_trees/gradient_boosted_trees.cc:1790] "goss_alpha" set but "sampling_method" not equal to "GOSS".
2026-05-09 19:08:41.093573: W external/ydf/yggdrasil_decision_forests/learner/gradient_boosted_trees/gradient_boosted_trees.cc:1800] "goss_beta" set but "sampling_method" not equal to "GOSS".
2026-05-09 19:08:41.093580: W external/ydf/yggdrasil_decision_forests/learner/gradient_boosted_trees/gradient_boosted_trees.cc:1814] "selective_gradient_boosting_ratio" set but "sampling_method" not equal to "SELGB".
2026-05-09 19:08:41.093861: I external/ydf/yggdrasil_decision_forests/learner/gradient_boosted_trees/gradient_boosted_trees.cc:452] Default loss set to SQUARED_ERROR
2026-05-09 19:08:41.093879: I external/ydf/yggdrasil_decision_forests/learner/gradient_boosted_trees/gradient_boosted_trees.cc:1077] Training gradient boosted tree on 143074 example(s) and 14 feature(s).
2026-05-09 19:08:41.109033: 

Model trained in 0:02:21.051566
Compiling model...


2026-05-09 19:11:02.059117: I external/ydf/yggdrasil_decision_forests/learner/gradient_boosted_trees/gradient_boosted_trees.cc:1568] Create final snapshot of the model at iteration 165
2026-05-09 19:11:02.063447: I external/ydf/yggdrasil_decision_forests/learner/gradient_boosted_trees/gradient_boosted_trees.cc:247] Truncates the model to 136 tree(s) i.e. 136  iteration(s).
2026-05-09 19:11:02.063699: I external/ydf/yggdrasil_decision_forests/learner/gradient_boosted_trees/gradient_boosted_trees.cc:309] Final model num-trees:136 valid-loss:802255.875000 valid-rmse:802255.875000
[INFO 2026-05-09T19:11:02.085528639-04:00 kernel.cc:1214] Loading model from path /tmp/tmp5_gmw3vn/model/ with prefix 42e7045809744a9d
[INFO 2026-05-09T19:11:02.101693627-04:00 abstract_model.cc:1311] Engine "GradientBoostedTreesQuickScorerExtended" built
[INFO 2026-05-09T19:11:02.10174256-04:00 kernel.cc:1046] Use fast generic engine


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: could not get source code
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: could not get source code
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: could not get source code
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Model compiled.


In [ ]:
# Evaluar el modelo
resultados = model.evaluate(test_df, return_dict=True)
print("Métricas disponibles:", resultados.keys())

# 1. Tomar un lote de datos del set de prueba
for features, labels in test_df:
    # 2. Hacer la predicción
    preds = model.predict(features)
    
    # 3. Ver los primeros 5 resultados
    print("Primeras 5 Predicciones vs Realidad:")
    print(f"Predicho: ${preds} | Real: ${labels[i]:,.0f}")
    
    # 4. Calcular el RMSE manual de este lote
    rmse_manual = np.sqrt(np.mean((preds.flatten() - labels.numpy())**2))
    print(f"\nRMSE Manual del lote: ${rmse_manual:,.0f}")

36/36 [==============================] - 0s 5ms/step - loss: 0.0000e+00
Métricas disponibles: dict_keys(['loss'])
32/32 [==============================] - 0s 896us/step
Primeras 5 Predicciones vs Realidad:
Predicho: $87,134 | Real: $30,449

RMSE Manual del lote: $291,697
32/32 [==============================] - 0s 931us/step
Primeras 5 Predicciones vs Realidad:
Predicho: $54,186 | Real: $41,429

RMSE Manual del lote: $123,829
32/32 [==============================] - 0s 901us/step
Primeras 5 Predicciones vs Realidad:
Predicho: $76,106 | Real: $73,965

RMSE Manual del lote: $114,212
32/32 [==============================] - 0s 918us/step
Primeras 5 Predicciones vs Realidad:
Predicho: $61,963 | Real: $25,000

RMSE Manual del lote: $97,433
32/32 [==============================] - 0s 881us/step
Primeras 5 Predicciones vs Realidad:
Predicho: $88,363 | Real: $47,000

RMSE Manual del lote: $316,525
32/32 [==============================] - 0s 908us/step
Primeras 5 Predicciones vs Realidad:
Predi

# Random Forest Regressor
https://www.tensorflow.org/decision_forests/api_docs/python/tfdf/keras/RandomForestModel

In [12]:
model_rf = tfdf.keras.RandomForestModel(
    task = tfdf.keras.Task.REGRESSION,
    num_trees = 500, 
    max_depth = 16, 
    growing_strategy = "BEST_FIRST_GLOBAL",
)

model_rf.fit(train_df) 

Use /tmp/tmp3kd7wlzm as temporary training directory
Reading training dataset...
Training dataset read in 0:00:00.968487. Found 143074 examples.
Training model...
Model trained in 0:00:38.337546
Compiling model...


[INFO 2026-05-09T20:34:22.661681666-04:00 kernel.cc:1214] Loading model from path /tmp/tmp3kd7wlzm/model/ with prefix 95004ca6ce5f4c9f
[INFO 2026-05-09T20:34:22.692031953-04:00 decision_forest.cc:661] Model loaded with 291 root(s), 17751 node(s), and 15 input feature(s).
[INFO 2026-05-09T20:34:22.692051317-04:00 abstract_model.cc:1311] Engine "RandomForestGeneric" built
[INFO 2026-05-09T20:34:22.692062968-04:00 kernel.cc:1046] Use fast generic engine


Model compiled.


In [13]:
# 1. Listas para acumular absolutamente todo
todos_los_predichos = []
todos_los_reales = []

print(f"Calculando predicciones para las {len(test_df):,} filas del set de prueba...")

# 2. Recorrer el dataset completo (sin 'break')
for features, labels in test_df:
    # Predicción silenciosa del lote actual
    preds = model_rf.predict(features, verbose=0)
    
    # Guardar los valores en nuestras listas globales
    todos_los_predichos.extend(preds.flatten())
    todos_los_reales.extend(labels.numpy())

# 3. Convertir a arrays de Numpy para cálculos rápidos
todos_los_predichos = np.array(todos_los_predichos)
todos_los_reales = np.array(todos_los_reales)

# 4. Calcular métricas finales sobre el TOTAL
rmse_final = np.sqrt(np.mean((todos_los_predichos - todos_los_reales)**2))
mae_final = np.mean(np.abs(todos_los_predichos - todos_los_reales))

# 5. Mostrar resultados para tu Solemne
print("\n" + "="*40)
print("   RESULTADOS GLOBALES DEL MODELO")
print("="*40)
print(f"Total de datos evaluados: {len(todos_los_reales):,}")
print(f"RMSE Global: ${rmse_final:,.0f} pesos")
print(f"MAE Global:  ${mae_final:,.0f} pesos")
print("="*40)

Calculando predicciones para las 36 filas del set de prueba...

   RESULTADOS GLOBALES DEL MODELO
Total de datos evaluados: 35,768
RMSE Global: $591,708 pesos
MAE Global:  $40,220 pesos


# Deep Neural Network

In [32]:
y = df['price']
df = df.drop(['price'], axis=1)
x = df

# Separar los datos en train, validation, testing
x_train_full, x_test, y_train_full, y_test = train_test_split(x, y, test_size=0.01, random_state=42) #This return a data frame type
x_train, x_val, y_train, y_val = train_test_split(x_train_full, y_train_full, test_size=0.01, random_state=42)

# Crear Dataset para optimizar TF
train_ds = tf.data.Dataset.from_tensor_slices((x_train.values, y_train.values))
val_ds = tf.data.Dataset.from_tensor_slices((x_val.values, y_val.values))

# Optimizar Datasets
train_ds = train_ds.shuffle(buffer_size=10000).batch(32).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.batch(32).prefetch(tf.data.AUTOTUNE)

ValueError: Failed to convert a NumPy array to a Tensor (Unsupported object type int).

In [ ]:
input_columns = df.shape[1] - 1
model_mlp = keras.Sequential(
    [
        layers.Dense(30, activation="relu", input_shape=(input_columns), name="layer1"),
        layers.Dense(30, activation="relu", name="layer2"),
        layers.Dense(30, name="layer3"),
        layers.Dense(1)
    ]
)
x = tf.ones((1, 4))
y = model_mlp(x)
print("Number of weights after calling the model:", len(model_mlp.weights))  # 6
model_mlp.summary()

Number of weights after calling the model: 8
Model: "sequential_5"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 layer1 (Dense)              (1, 30)                   150       
                                                                 
 layer2 (Dense)              (1, 30)                   930       
                                                                 
 layer3 (Dense)              (1, 30)                   930       
                                                                 
 dense (Dense)               (1, 1)                    31        
                                                                 
Total params: 2,041
Trainable params: 2,041
Non-trainable params: 0
_________________________________________________________________


In [30]:
model_mlp.compile(
    optimizer='adam',
    loss='mean_squared_error',
    metrics=['mae', tf.keras.metrics.RootMeanSquaredError()],
)

model_mlp.fit(
    train_ds,
    validation_data = val_ds,
    epochs=50,
    verbose = 1
)

NameError: name 'train_ds' is not defined